### If you get an error here, it means you need to run the prepare_data.ipynb file first

In [1]:
import json
import numpy as np
import pandas as pd

from rnanorm import TMM
from pathlib import Path


description_path = Path('description.json')
if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    data = json.load(f)

## Preprocessing steps

1. **Load data** — Read the dataset and split columns into clinical variables and raw gene-level counts.

2. **Robust-expression filter (defined on raw counts)** — Keep genes with the 75th percentile of raw counts across samples \(\ge 100\) (i.e., expressed at \(\ge 100\) counts in at least 25% of samples).

3. **TMM normalization** — Apply TMM to the full gene count matrix to compute normalization factors correcting for library size and RNA composition; obtain normalized counts.

4. **Apply gene filter and log-transform** — Subset to the robustly expressed genes and apply \(\log(x+1)\).

5. <strike>**Z-score standardization** — For each gene, center and scale across samples to mean 0 and standard deviation 1.</strike>

In [2]:
data_file_name = data['data_file_names'][0]

num_clinical_features = data['data_summary']['clinical_features']

df = pd.read_csv(data_file_name)


clinical_cols = df.columns[:num_clinical_features]
gene_cols = df.columns[num_clinical_features:]

raw_counts = df[gene_cols]
print('Before preprocessing:\n - n_genes: %d' % (df[gene_cols].shape[1]))

# create mask of robustly expressed genes
robus_genes_mask = np.percentile(df[gene_cols], 75, axis=0) >= 100

# normalize counts
df[gene_cols] = TMM().set_output(transform="pandas").fit_transform(df[gene_cols])

# apply robust gene mask
gene_cols = df[gene_cols].columns[robus_genes_mask].tolist()
df = df[list(clinical_cols) + gene_cols]

# # log transform and standardize (genes only)
df[gene_cols] = np.log(df[gene_cols] + 1)
# df[gene_cols] = (df[gene_cols] - df[gene_cols].mean(axis=0)) / df[gene_cols].std(axis=0)

print("\nAfter preprocessing:\n - n_genes: %d" % (df[gene_cols].shape[1]))

transformed_data_file_name = data_file_name.replace("_raw_counts_NCBI", "_preprocessed")


Before preprocessing:
 - n_genes: 39376


/Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/.venv/lib/python3.14/site-packages/rnanorm/methods/between_sample.py:101: RuntimeWarning: divide by zero encountered in divide
  factors = factors / self.geometric_mean_
/Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/.venv/lib/python3.14/site-packages/rnanorm/methods/between_sample.py:101: RuntimeWarning: invalid value encountered in divide
  factors = factors / self.geometric_mean_



After preprocessing:
 - n_genes: 81


In [3]:
url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
df_gene_annotations = df_gene_annotations[df_gene_annotations["Status"] == "active"]
df_gene_annotations = df_gene_annotations[df_gene_annotations["GeneType"] == "protein-coding"]
df_gene_annotations = df_gene_annotations.reset_index(drop=True)

protein_coding_genes = set(df_gene_annotations["Symbol"].dropna())
gene_cols = [gene for gene in gene_cols if gene in protein_coding_genes]
df = df[list(clinical_cols) + gene_cols]

print("\nAfter protein-coding filter:\n - n_genes: %d" % len(gene_cols))

df.to_csv(transformed_data_file_name, index=False)

if transformed_data_file_name not in data['data_file_names']:
    data['data_file_names'].append(transformed_data_file_name)

with open(description_path, 'w') as f:
    json.dump(data, f, indent=4)


After protein-coding filter:
 - n_genes: 78


/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_14644/4029965179.py:2: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


## Survival variables encodings

Authors encode survival variables in various ways, here we have just a check if all the survival variables are already numeric. If they are not, we convert them by hand.


In [4]:
df_ = df.copy()

endpoints = []
for endpoint in data["survival-endpoints"]:
    endpoint_time = endpoint["time_var"]["var_name"]
    endpoint_event = endpoint["event_var"]["var_name"]

    if endpoint_time == "unknown" or endpoint_event == "unknown":
        continue

    # Replace "Not available" and similar missing-value strings with NaN
    for col in [endpoint_time, endpoint_event]:
        df_[col] = df_[col].replace(["Not available", "NA", "N/A", "not available", "-"], pd.NA)


    # make sure time and event are numeric
    df_[endpoint_time] = pd.to_numeric(df_[endpoint_time])
    # df_[endpoint_event] = df_[endpoint_event].map({"Dead": 1, "Y": 1, "Alive": 0, "N": 0})
    df_[endpoint_event] = pd.to_numeric(df_[endpoint_event])
    
    endpoints.append((endpoint_time, endpoint_event))


# Flatten endpoints and use it to order survival columns
surv_cols = [col for tup in endpoints for col in tup]
other_cols = [col for col in df_.columns if col not in surv_cols]
ordered_cols = surv_cols + other_cols

df_[ordered_cols].to_csv(transformed_data_file_name, index=False)

In [5]:
df_[ordered_cols]

,dmfs time,dmfs event,sample_title,gender,sample type,irdm subtype,pam50 subtype,immunity group,proliferation group,er status rna,...,MIA,MYBL2,UBE2C,AURKA,MMP11,SF3A1,CCDC157,APOBEC3G,IL2RG,SH2D1A
0,1.916667,0,SH_TS_BC113,female,Solid tumor,Basal,Basal,iweak,high,ER-,...,8.669655,0.000000,10.785072,9.701948,0.000000,0.000000,0.000000,0.000000,7.698076,0.000000
1,6.750000,0,SH_TS_BC119,female,Solid tumor,HER2E,HER2E,istrong,high,ER-,...,5.828747,0.000000,9.761326,9.029801,0.000000,0.000000,0.000000,0.000000,8.699401,6.999987
2,3.416667,0,SH_TS_BC146,female,Solid tumor,Basal,Basal,iweak,high,ER+,...,7.661877,0.000000,10.722630,9.294487,0.000000,0.000000,0.000000,0.000000,7.882371,5.965590
3,9.750000,0,SH_TS_BC147,female,Solid tumor,LumB,LumB,iweak,high,ER+,...,6.023581,0.000000,9.533534,8.878157,0.000000,0.000000,0.000000,0.000000,2.705913,7.212216
4,9.083333,0,SH_TS_BC150,female,Solid tumor,LumA,LumA,istrong,low,ER+,...,7.671198,0.000000,8.358780,8.501992,0.000000,0.000000,0.000000,0.000000,8.094774,7.722999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,5.330000,0,UNC_TGS_BC_96,female,Solid tumor,LumA,LumA,iweak,low,ER+,...,9.460170,8.471764,8.212315,7.334755,9.595315,5.431419,5.431419,8.634014,7.424594,7.810066
115,4.830000,0,UNC_TGS_BC_97,female,Solid tumor,LumA,LumA,iweak,low,ER+,...,7.353910,7.750213,8.478476,6.600858,9.847544,7.619820,7.619820,8.322913,6.902290,7.133594
116,5.420000,0,UNC_TGS_BC_99,female,Solid tumor,Basal,Basal,iweak,high,ER-,...,8.699991,9.888811,9.354256,8.092750,10.660153,6.964177,6.964177,7.600219,6.045067,0.000000
117,13.750000,0,UNC_TGS_BC_Y23,female,Solid tumor,Basal,Basal,istrong,high,ER-,...,7.373944,9.973570,9.395905,8.366720,11.302058,4.578121,4.578121,9.304709,6.367206,6.889098
